# P&L Engineering and KPI Framework

**Prepared for:** CFO & Operations Manager

**Date:** April 2026

**Objective:** Implement comprehensive P&L structure, compute key performance indicators, establish performance scoring methodology, and generate executive-level reporting for Java House chain optimization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Data Loading and Preparation

Load operational and master data, then prepare for P&L and KPI analysis.

In [ ]:
# Load data
master = pd.read_csv('../data/restaurant_master.csv')
operations = pd.read_csv('../data/daily_operations.csv')

print("Master dataset shape:", master.shape)
print("Operations dataset shape:", operations.shape)

# Convert date and merge
operations['date'] = pd.to_datetime(operations['date'])
data = pd.merge(operations, master, on='restaurant_id')

# Add time dimensions
data['month'] = data['date'].dt.month
data['year'] = data['date'].dt.year
data['quarter'] = data['date'].dt.quarter
data['day_of_week'] = data['date'].dt.day_name()
data['is_weekend'] = data['day_of_week'].isin(['Saturday', 'Sunday'])

print("\nMerged dataset shape:", data.shape)
print("Date range:", data['date'].min(), "to", data['date'].max())

## 2. P&L Structure Definition

Define comprehensive profit and loss structure with hierarchical cost categorization.

In [ ]:
class PLStructure:
    """Profit & Loss structure for restaurant operations"""
    
    def __init__(self, data):
        self.data = data.copy()
        self._calculate_pl_components()
    
    def _calculate_pl_components(self):
        """Calculate all P&L components"""
        # Revenue
        self.data['total_revenue'] = self.data['revenue']
        
        # Cost of Goods Sold (COGS)
        self.data['cogs'] = self.data['food_cost']
        self.data['gross_profit'] = self.data['total_revenue'] - self.data['cogs']
        self.data['gross_margin'] = self.data['gross_profit'] / self.data['total_revenue']
        
        # Operating Expenses
        self.data['opex_total'] = (
            self.data['staff_cost'] + 
            self.data['rent'] + 
            self.data['operating_expenses']
        )
        
        # Operating Expense Breakdown
        self.data['opex_labor'] = self.data['staff_cost']
        self.data['opex_rent'] = self.data['rent']
        self.data['opex_other'] = self.data['operating_expenses']
        
        # Operating Profit (EBIT)
        self.data['operating_profit'] = self.data['gross_profit'] - self.data['opex_total']
        self.data['operating_margin'] = self.data['operating_profit'] / self.data['total_revenue']
        
        # Net Profit (maintains existing calculation for consistency)
        self.data['net_profit_margin'] = self.data['net_profit'] / self.data['total_revenue']
        
        # Cost percentages
        cost_components = ['cogs', 'opex_labor', 'opex_rent', 'opex_other']
        for component in cost_components:
            self.data[f'{component}_pct'] = self.data[component] / self.data['total_revenue']
    
    def get_pl_summary(self, groupby_cols=None):
        """Get P&L summary by specified grouping"""
        if groupby_cols is None:
            # Overall summary
            summary = self.data.agg({
                'total_revenue': 'sum',
                'cogs': 'sum',
                'gross_profit': 'sum',
                'opex_total': 'sum',
                'operating_profit': 'sum',
                'net_profit': 'sum'
            }).to_frame('Total')
        else:
            summary = self.data.groupby(groupby_cols).agg({
                'total_revenue': 'sum',
                'cogs': 'sum',
                'gross_profit': 'sum',
                'opex_total': 'sum',
                'operating_profit': 'sum',
                'net_profit': 'sum'
            })
        
        # Calculate margins
        summary['gross_margin'] = summary['gross_profit'] / summary['total_revenue']
        summary['operating_margin'] = summary['operating_profit'] / summary['total_revenue']
        summary['net_margin'] = summary['net_profit'] / summary['total_revenue']
        
        return summary.round(2)

# Initialize P&L structure
pl_engine = PLStructure(data)

# Overall P&L summary
overall_pl = pl_engine.get_pl_summary()
print("Overall P&L Summary (KES):")
print(overall_pl)

# Branch-level P&L
branch_pl = pl_engine.get_pl_summary(['restaurant_name', 'location_tier'])
print("\nBranch P&L Summary (Top 5 by Revenue):")
print(branch_pl.nlargest(5, 'total_revenue'))

## 3. KPI Framework

Define and compute comprehensive set of key performance indicators across financial, operational, and efficiency dimensions.

In [ ]:
class KPIFramework:
    """Comprehensive KPI framework for restaurant performance evaluation"""
    
    def __init__(self, data):
        self.data = data.copy()
        self.kpis = {}
        self._calculate_financial_kpis()
        self._calculate_operational_kpis()
        self._calculate_efficiency_kpis()
        self._calculate_growth_kpis()
    
    def _calculate_financial_kpis(self):
        """Calculate financial performance KPIs"""
        # Profitability KPIs
        self.kpis['gross_margin'] = self.data['gross_profit'] / self.data['revenue']
        self.kpis['operating_margin'] = self.data['operating_profit'] / self.data['revenue']
        self.kpis['net_profit_margin'] = self.data['net_profit'] / self.data['revenue']
        
        # Revenue KPIs
        self.kpis['revenue_per_customer'] = self.data['revenue'] / self.data['customers']
        self.kpis['avg_order_value'] = self.data['avg_order_value']
        
        # Cost KPIs
        self.kpis['cogs_percentage'] = self.data['food_cost'] / self.data['revenue']
        self.kpis['labor_cost_percentage'] = self.data['staff_cost'] / self.data['revenue']
        self.kpis['rent_percentage'] = self.data['rent'] / self.data['revenue']
        self.kpis['opex_percentage'] = self.data['operating_expenses'] / self.data['revenue']
    
    def _calculate_operational_kpis(self):
        """Calculate operational performance KPIs"""
        # Capacity and utilization
        self.kpis['daily_customers'] = self.data['customers']
        self.kpis['customer_satisfaction_proxy'] = self.data['base_rating']  # Using base rating as proxy
        
        # Staff efficiency
        self.kpis['revenue_per_staff'] = self.data['revenue'] / self.data['staff_count']
        self.kpis['profit_per_staff'] = self.data['net_profit'] / self.data['staff_count']
        
        # Location efficiency
        self.kpis['revenue_per_sqft_proxy'] = self.data['revenue'] / self.data['base_rent']  # Rent as size proxy
        self.kpis['efficiency_index'] = self.data['efficiency_score']
    
    def _calculate_efficiency_kpis(self):
        """Calculate efficiency and productivity KPIs"""
        # Break-even analysis
        total_costs = self.data['food_cost'] + self.data['staff_cost'] + self.data['rent'] + self.data['operating_expenses']
        self.kpis['contribution_margin'] = (self.data['revenue'] - self.data['food_cost'] - self.data['staff_cost']) / self.data['revenue']
        
        # Productivity ratios
        self.kpis['asset_turnover_proxy'] = self.data['revenue'] / self.data['base_rent']  # Revenue to rent ratio
        self.kpis['cost_efficiency_ratio'] = self.data['net_profit'] / total_costs
        
        # Performance ratios
        self.kpis['return_on_investment_proxy'] = self.data['net_profit'] / self.data['base_rent']
    
    def _calculate_growth_kpis(self):
        """Calculate growth and trend KPIs (monthly basis)"""
        # Monthly aggregations for growth calculations
        monthly_data = self.data.groupby(['restaurant_id', 'year', 'month']).agg({
            'revenue': 'sum',
            'customers': 'sum',
            'net_profit': 'sum'
        }).reset_index()
        
        monthly_data['period'] = monthly_data['year'].astype(str) + '-' + monthly_data['month'].astype(str).str.zfill(2)
        monthly_data = monthly_data.sort_values(['restaurant_id', 'year', 'month'])
        
        # Calculate month-over-month growth
        monthly_data['revenue_mom_growth'] = monthly_data.groupby('restaurant_id')['revenue'].pct_change()
        monthly_data['customers_mom_growth'] = monthly_data.groupby('restaurant_id')['customers'].pct_change()
        monthly_data['profit_mom_growth'] = monthly_data.groupby('restaurant_id')['net_profit'].pct_change()
        
        # Merge growth KPIs back to daily data (using last available growth rate per restaurant)
        latest_growth = monthly_data.groupby('restaurant_id').agg({
            'revenue_mom_growth': 'last',
            'customers_mom_growth': 'last',
            'profit_mom_growth': 'last'
        }).reset_index()
        
        # Add growth KPIs to main KPI set
        growth_dict = latest_growth.set_index('restaurant_id')[['revenue_mom_growth', 'customers_mom_growth', 'profit_mom_growth']].to_dict('index')
        
        for kpi_name in ['revenue_mom_growth', 'customers_mom_growth', 'profit_mom_growth']:
            self.kpis[kpi_name] = self.data['restaurant_id'].map(lambda x: growth_dict.get(x, {}).get(kpi_name, 0))
    
    def get_kpi_summary(self, groupby_cols=None, kpi_list=None):
        """Get KPI summary statistics"""
        if kpi_list is None:
            kpi_list = list(self.kpis.keys())
        
        summary_data = self.data.copy()
        for kpi_name, kpi_values in self.kpis.items():
            if kpi_name in kpi_list:
                summary_data[kpi_name] = kpi_values
        
        if groupby_cols is None:
            summary = summary_data[kpi_list].agg(['mean', 'std', 'min', 'max']).T
            summary.columns = ['Mean', 'Std', 'Min', 'Max']
        else:
            summary = summary_data.groupby(groupby_cols)[kpi_list].mean()
        
        return summary.round(4)

# Initialize KPI framework
kpi_engine = KPIFramework(pl_engine.data)

# Display KPI summary
print("KPI Framework Summary:")
kpi_summary = kpi_engine.get_kpi_summary()
print(kpi_summary)

# Key financial KPIs by location tier
key_kpis = ['gross_margin', 'operating_margin', 'net_profit_margin', 'revenue_per_customer', 'cogs_percentage']
tier_kpi_summary = kpi_engine.get_kpi_summary(['location_tier'], key_kpis)
print("\nKey KPIs by Location Tier:")
print(tier_kpi_summary)

## 4. Data Consistency Validation

Validate P&L calculations and KPI computations for logical consistency and business rules compliance.

In [ ]:
class DataValidator:
    """Data consistency and validation checks for P&L and KPIs"""
    
    def __init__(self, data):
        self.data = data.copy()
        self.validation_results = {}
    
    def validate_pl_consistency(self):
        """Validate P&L mathematical consistency"""
        print("Validating P&L Consistency...")
        
        # Revenue decomposition check
        calculated_revenue = self.data['customers'] * self.data['avg_order_value']
        revenue_discrepancy = abs(self.data['revenue'] - calculated_revenue) > 0.01
        self.validation_results['revenue_calculation'] = {
            'discrepancies': revenue_discrepancy.sum(),
            'percentage': revenue_discrepancy.sum() / len(self.data) * 100
        }
        
        # Profit calculation check
        total_costs = self.data['food_cost'] + self.data['staff_cost'] + self.data['rent'] + self.data['operating_expenses']
        calculated_profit = self.data['revenue'] - total_costs
        profit_discrepancy = abs(self.data['net_profit'] - calculated_profit) > 0.01
        self.validation_results['profit_calculation'] = {
            'discrepancies': profit_discrepancy.sum(),
            'percentage': profit_discrepancy.sum() / len(self.data) * 100
        }
        
        # Margin consistency check
        margin_checks = {
            'gross_margin_range': ((self.data['gross_margin'] >= -0.5) & (self.data['gross_margin'] <= 1)).all(),
            'operating_margin_range': ((self.data['operating_margin'] >= -1) & (self.data['operating_margin'] <= 1)).all(),
            'net_margin_range': ((self.data['net_margin'] >= -1) & (self.data['net_margin'] <= 1)).all()
        }
        self.validation_results['margin_ranges'] = margin_checks
        
        return self.validation_results
    
    def validate_business_rules(self):
        """Validate against business rules and logical constraints"""
        print("Validating Business Rules...")
        
        rules = {
            'positive_revenue': (self.data['revenue'] > 0).all(),
            'positive_customers': (self.data['customers'] > 0).all(),
            'reasonable_customer_counts': ((self.data['customers'] >= 10) & (self.data['customers'] <= 1000)).mean() > 0.95,
            'reasonable_order_values': ((self.data['avg_order_value'] >= 300) & (self.data['avg_order_value'] <= 1500)).mean() > 0.95,
            'cost_not_exceed_revenue': ((self.data['food_cost'] + self.data['staff_cost'] + self.data['rent'] + self.data['operating_expenses']) <= self.data['revenue'] * 1.5).all(),
            'positive_costs': (self.data[['food_cost', 'staff_cost', 'rent', 'operating_expenses']] >= 0).all().all()
        }
        
        self.validation_results['business_rules'] = rules
        return rules
    
    def validate_kpi_reasonableness(self):
        """Validate KPI values are within reasonable ranges"""
        print("Validating KPI Reasonableness...")
        
        kpi_ranges = {
            'gross_margin': {'min': 0.2, 'max': 0.8, 'target': 0.6},
            'operating_margin': {'min': -0.2, 'max': 0.4, 'target': 0.15},
            'net_profit_margin': {'min': -0.3, 'max': 0.3, 'target': 0.08},
            'cogs_percentage': {'min': 0.2, 'max': 0.5, 'target': 0.35},
            'labor_cost_percentage': {'min': 0.05, 'max': 0.25, 'target': 0.15},
            'rent_percentage': {'min': 0.02, 'max': 0.15, 'target': 0.08}
        }
        
        kpi_validation = {}
        for kpi, ranges in kpi_ranges.items():
            if kpi in self.data.columns:
                mean_val = self.data[kpi].mean()
                within_range = (mean_val >= ranges['min']) and (mean_val <= ranges['max'])
                kpi_validation[kpi] = {
                    'mean': mean_val,
                    'within_range': within_range,
                    'target': ranges['target']
                }
        
        self.validation_results['kpi_ranges'] = kpi_validation
        return kpi_validation
    
    def generate_validation_report(self):
        """Generate comprehensive validation report"""
        self.validate_pl_consistency()
        self.validate_business_rules()
        self.validate_kpi_reasonableness()
        
        print("\n=== DATA VALIDATION REPORT ===")
        
        # P&L Consistency
        print("\nP&L Consistency:")
        print(f"Revenue calculation discrepancies: {self.validation_results['revenue_calculation']['discrepancies']} ({self.validation_results['revenue_calculation']['percentage']:.2f}%)")
        print(f"Profit calculation discrepancies: {self.validation_results['profit_calculation']['discrepancies']} ({self.validation_results['profit_calculation']['percentage']:.2f}%)")
        
        # Business Rules
        print("\nBusiness Rules Compliance:")
        for rule, passed in self.validation_results['business_rules'].items():
            status = "PASS" if passed else "FAIL"
            print(f"{rule}: {status}")
        
        # KPI Ranges
        print("\nKPI Range Validation:")
        for kpi, validation in self.validation_results['kpi_ranges'].items():
            status = "PASS" if validation['within_range'] else "FAIL"
            print(f"{kpi}: {validation['mean']:.3f} (Target: {validation['target']:.3f}) - {status}")
        
        return self.validation_results

# Run validation
validator = DataValidator(pl_engine.data)
validation_report = validator.generate_validation_report()

## 5. Performance Scoring Framework

Develop multi-dimensional performance scoring system for restaurant evaluation and benchmarking.

In [ ]:
class PerformanceScorer:
    """Multi-dimensional performance scoring system"""
    
    def __init__(self, data, kpi_engine):
        self.data = data.copy()
        self.kpi_engine = kpi_engine
        self.scores = {}
        self.weights = self._define_weights()
        self._calculate_dimension_scores()
        self._calculate_overall_score()
    
    def _define_weights(self):
        """Define weights for different performance dimensions"""
        return {
            'financial': {
                'net_profit_margin': 0.25,
                'operating_margin': 0.20,
                'gross_margin': 0.15,
                'revenue_per_customer': 0.15,
                'return_on_investment_proxy': 0.25
            },
            'operational': {
                'efficiency_index': 0.25,
                'revenue_per_staff': 0.20,
                'customer_satisfaction_proxy': 0.15,
                'daily_customers': 0.15,
                'cost_efficiency_ratio': 0.25
            },
            'growth': {
                'revenue_mom_growth': 0.40,
                'customers_mom_growth': 0.30,
                'profit_mom_growth': 0.30
            }
        }
    
    def _calculate_dimension_scores(self):
        """Calculate scores for each performance dimension"""
        
        # Financial dimension
        financial_kpis = list(self.weights['financial'].keys())
        financial_scores = []
        
        for kpi in financial_kpis:
            if kpi in self.kpi_engine.kpis:
                # Normalize to 0-100 scale (higher is better)
                kpi_values = self.kpi_engine.kpis[kpi]
                if kpi in ['net_profit_margin', 'operating_margin', 'gross_margin', 'revenue_per_customer', 'return_on_investment_proxy']:
                    # Higher values are better
                    normalized = (kpi_values - kpi_values.min()) / (kpi_values.max() - kpi_values.min()) * 100
                else:
                    # For cost ratios, lower is better (invert)
                    normalized = (1 - (kpi_values - kpi_values.min()) / (kpi_values.max() - kpi_values.min())) * 100
                
                weighted_score = normalized * self.weights['financial'][kpi]
                financial_scores.append(weighted_score)
        
        self.scores['financial'] = sum(financial_scores)
        
        # Operational dimension
        operational_kpis = list(self.weights['operational'].keys())
        operational_scores = []
        
        for kpi in operational_kpis:
            if kpi in self.kpi_engine.kpis:
                kpi_values = self.kpi_engine.kpis[kpi]
                # Higher values are better for operational KPIs
                normalized = (kpi_values - kpi_values.min()) / (kpi_values.max() - kpi_values.min()) * 100
                weighted_score = normalized * self.weights['operational'][kpi]
                operational_scores.append(weighted_score)
        
        self.scores['operational'] = sum(operational_scores)
        
        # Growth dimension
        growth_kpis = list(self.weights['growth'].keys())
        growth_scores = []
        
        for kpi in growth_kpis:
            if kpi in self.kpi_engine.kpis:
                kpi_values = self.kpi_engine.kpis[kpi]
                # Handle negative growth rates
                normalized = np.maximum(0, (kpi_values + 1) / 2 * 100)  # Convert to 0-100 scale
                weighted_score = normalized * self.weights['growth'][kpi]
                growth_scores.append(weighted_score)
        
        self.scores['growth'] = sum(growth_scores)
    
    def _calculate_overall_score(self):
        """Calculate overall performance score"""
        dimension_weights = {'financial': 0.5, 'operational': 0.3, 'growth': 0.2}
        self.scores['overall'] = (
            self.scores['financial'] * dimension_weights['financial'] +
            self.scores['operational'] * dimension_weights['operational'] +
            self.scores['growth'] * dimension_weights['growth']
        )
    
    def get_performance_summary(self, groupby_cols=None):
        """Get performance scores summary"""
        score_data = self.data.copy()
        score_data['financial_score'] = self.scores['financial']
        score_data['operational_score'] = self.scores['operational']
        score_data['growth_score'] = self.scores['growth']
        score_data['overall_score'] = self.scores['overall']
        
        if groupby_cols is None:
            summary = score_data[['financial_score', 'operational_score', 'growth_score', 'overall_score']].mean()
        else:
            summary = score_data.groupby(groupby_cols)[['financial_score', 'operational_score', 'growth_score', 'overall_score']].mean()
        
        return summary.round(2)
    
    def get_performance_bands(self):
        """Categorize performance into bands"""
        score_data = self.data.copy()
        score_data['overall_score'] = self.scores['overall']
        
        # Define performance bands
        conditions = [
            score_data['overall_score'] >= 80,
            score_data['overall_score'] >= 60,
            score_data['overall_score'] >= 40,
            score_data['overall_score'] >= 20
        ]
        choices = ['Excellent', 'Good', 'Average', 'Poor']
        score_data['performance_band'] = np.select(conditions, choices, default='Critical')
        
        return score_data.groupby('restaurant_id')['performance_band'].first()

# Initialize performance scorer
scorer = PerformanceScorer(pl_engine.data, kpi_engine)

# Performance summary
performance_summary = scorer.get_performance_summary()
print("Overall Performance Scores:")
print(performance_summary)

# Branch-level performance
branch_performance = scorer.get_performance_summary(['restaurant_name', 'location_tier'])
print("\nBranch Performance Scores (Top 5):")
print(branch_performance.nlargest(5, 'overall_score'))

# Performance bands
performance_bands = scorer.get_performance_bands()
print("\nPerformance Band Distribution:")
print(performance_bands.value_counts())

## 6. Performance Segmentation

Apply clustering analysis to identify distinct performance segments and strategic groups.

In [ ]:
class PerformanceSegmenter:
    """Unsupervised segmentation of restaurant performance"""
    
    def __init__(self, data, kpi_engine, n_clusters=4):
        self.data = data.copy()
        self.kpi_engine = kpi_engine
        self.n_clusters = n_clusters
        self.segmentation_features = self._select_features()
        self.segments = None
        self.segment_profiles = None
        self._perform_segmentation()
    
    def _select_features(self):
        """Select key features for segmentation"""
        return [
            'net_profit_margin', 'operating_margin', 'gross_margin',
            'revenue_per_customer', 'cogs_percentage', 'labor_cost_percentage',
            'efficiency_index', 'revenue_per_staff', 'daily_customers',
            'revenue_mom_growth', 'customers_mom_growth'
        ]
    
    def _perform_segmentation(self):
        """Perform K-means clustering on performance features"""
        # Prepare feature matrix
        feature_data = []
        for feature in self.segmentation_features:
            if feature in self.kpi_engine.kpis:
                feature_data.append(self.kpi_engine.kpis[feature])
        
        if not feature_data:
            raise ValueError("No valid features found for segmentation")
        
        X = np.column_stack(feature_data)
        
        # Handle missing values
        X = np.nan_to_num(X, nan=0)
        
        # Standardize features
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        
        # Perform clustering
        kmeans = KMeans(n_clusters=self.n_clusters, random_state=42, n_init=10)
        self.segments = kmeans.fit_predict(X_scaled)
        
        # Create segment profiles
        segment_data = self.data.copy()
        segment_data['segment'] = self.segments
        
        self.segment_profiles = segment_data.groupby('segment').agg({
            'revenue': 'mean',
            'net_profit': 'mean',
            'customers': 'mean',
            'net_profit_margin': 'mean',
            'efficiency_score': 'mean',
            'restaurant_id': 'count'
        }).round(2)
        
        # Add segment names based on characteristics
        self._name_segments()
    
    def _name_segments(self):
        """Assign descriptive names to segments based on their characteristics"""
        segment_names = {}
        
        for segment_id, profile in self.segment_profiles.iterrows():
            if profile['net_profit_margin'] > 0.1 and profile['revenue'] > self.segment_profiles['revenue'].quantile(0.75):
                segment_names[segment_id] = 'High-Performing Champions'
            elif profile['net_profit_margin'] > 0.05 and profile['efficiency_score'] > self.segment_profiles['efficiency_score'].quantile(0.5):
                segment_names[segment_id] = 'Efficient Operators'
            elif profile['net_profit_margin'] < 0 and profile['revenue'] < self.segment_profiles['revenue'].quantile(0.25):
                segment_names[segment_id] = 'Underperforming Locations'
            else:
                segment_names[segment_id] = 'Balanced Performers'
        
        self.segment_profiles['segment_name'] = self.segment_profiles.index.map(segment_names)
    
    def get_segment_summary(self):
        """Get segment profiles and characteristics"""
        return self.segment_profiles
    
    def get_restaurant_segments(self):
        """Get segment assignment for each restaurant"""
        segment_data = self.data.copy()
        segment_data['segment'] = self.segments
        segment_data['segment_name'] = segment_data['segment'].map(self.segment_profiles['segment_name'])
        
        return segment_data.groupby('restaurant_id')[['segment', 'segment_name']].first()
    
    def analyze_segment_differences(self):
        """Analyze key differences between segments"""
        analysis = {}
        
        # Revenue and profit differences
        analysis['revenue_range'] = {
            'min': self.segment_profiles['revenue'].min(),
            'max': self.segment_profiles['revenue'].max(),
            'spread': self.segment_profiles['revenue'].max() / self.segment_profiles['revenue'].min()
        }
        
        analysis['profit_range'] = {
            'min': self.segment_profiles['net_profit'].min(),
            'max': self.segment_profiles['net_profit'].max(),
            'spread': abs(self.segment_profiles['net_profit'].max() / self.segment_profiles['net_profit'].min())
        }
        
        # Location tier distribution by segment
        segment_data = self.data.copy()
        segment_data['segment'] = self.segments
        segment_data['segment_name'] = segment_data['segment'].map(self.segment_profiles['segment_name'])
        
        tier_distribution = segment_data.groupby(['segment_name', 'location_tier']).size().unstack(fill_value=0)
        analysis['tier_distribution'] = tier_distribution
        
        return analysis

# Perform segmentation
segmenter = PerformanceSegmenter(pl_engine.data, kpi_engine, n_clusters=4)

# Segment profiles
segment_summary = segmenter.get_segment_summary()
print("Performance Segments:")
print(segment_summary)

# Restaurant segment assignments
restaurant_segments = segmenter.get_restaurant_segments()
print("\nRestaurant Segment Distribution:")
print(restaurant_segments['segment_name'].value_counts())

# Segment analysis
segment_analysis = segmenter.analyze_segment_differences()
print("\nSegment Analysis:")
print(f"Revenue spread across segments: {segment_analysis['revenue_range']['spread']:.1f}x")
print(f"Profit spread across segments: {segment_analysis['profit_range']['spread']:.1f}x")

# Visualization
plt.figure(figsize=(12, 8))
colors = ['blue', 'green', 'red', 'orange']

for i, (segment_id, profile) in enumerate(segment_summary.iterrows()):
    plt.scatter(profile['revenue'], profile['net_profit'], 
               s=profile['restaurant_id']*20, alpha=0.7,
               c=colors[i], label=f"{profile['segment_name']} (n={profile['restaurant_id']})")

plt.xlabel('Average Daily Revenue (KES)')
plt.ylabel('Average Daily Profit (KES)')
plt.title('Performance Segments: Revenue vs Profit')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 7. Executive Reporting

Generate comprehensive executive reports with key insights, recommendations, and strategic implications.

In [ ]:
class ExecutiveReport:
    """Executive reporting system for P&L and KPI insights"""
    
    def __init__(self, pl_engine, kpi_engine, scorer, segmenter):
        self.pl_engine = pl_engine
        self.kpi_engine = kpi_engine
        self.scorer = scorer
        self.segmenter = segmenter
    
    def generate_financial_summary(self):
        """Generate financial performance summary"""
        overall_pl = self.pl_engine.get_pl_summary()
        
        print("=== FINANCIAL PERFORMANCE SUMMARY ===")
        print(f"Total Revenue: KES {overall_pl.loc['Total', 'total_revenue']:,.0f}")
        print(f"Total Profit: KES {overall_pl.loc['Total', 'net_profit']:,.0f}")
        print(f"Overall Profit Margin: {overall_pl.loc['Total', 'net_margin']:.1%}")
        print(f"Gross Margin: {overall_pl.loc['Total', 'gross_margin']:.1%}")
        print(f"Operating Margin: {overall_pl.loc['Total', 'operating_margin']:.1%}")
        
        return overall_pl
    
    def generate_kpi_dashboard(self):
        """Generate KPI dashboard summary"""
        key_kpis = ['net_profit_margin', 'operating_margin', 'gross_margin', 
                   'revenue_per_customer', 'cogs_percentage', 'labor_cost_percentage']
        
        kpi_summary = self.kpi_engine.get_kpi_summary(kpi_list=key_kpis)
        
        print("\n=== KPI DASHBOARD ===")
        for kpi in key_kpis:
            mean_val = kpi_summary.loc[kpi, 'Mean']
            if 'margin' in kpi or 'percentage' in kpi:
                print(f"{kpi.replace('_', ' ').title()}: {mean_val:.1%}")
            else:
                print(f"{kpi.replace('_', ' ').title()}: KES {mean_val:,.0f}")
        
        return kpi_summary
    
    def generate_performance_insights(self):
        """Generate performance scoring insights"""
        performance_summary = self.scorer.get_performance_summary()
        performance_bands = self.scorer.get_performance_bands()
        
        print("\n=== PERFORMANCE INSIGHTS ===")
        print(f"Overall Performance Score: {performance_summary['overall_score']:.1f}/100")
        print(f"Financial Score: {performance_summary['financial_score']:.1f}/100")
        print(f"Operational Score: {performance_summary['operational_score']:.1f}/100")
        print(f"Growth Score: {performance_summary['growth_score']:.1f}/100")
        
        print("\nPerformance Band Distribution:")
        band_counts = performance_bands.value_counts()
        for band, count in band_counts.items():
            print(f"{band}: {count} locations ({count/len(performance_bands)*100:.1f}%)")
        
        return performance_summary, performance_bands
    
    def generate_segmentation_analysis(self):
        """Generate segmentation analysis report"""
        segment_summary = self.segmenter.get_segment_summary()
        
        print("\n=== SEGMENTATION ANALYSIS ===")
        for segment_id, profile in segment_summary.iterrows():
            print(f"\n{profile['segment_name']}:")
            print(f"  Locations: {profile['restaurant_id']}")
            print(f"  Avg Revenue: KES {profile['revenue']:,.0f}")
            print(f"  Avg Profit: KES {profile['net_profit']:,.0f}")
            print(f"  Profit Margin: {profile['net_profit_margin']:.1%}")
            print(f"  Efficiency Score: {profile['efficiency_score']:.2f}")
        
        return segment_summary
    
    def generate_strategic_recommendations(self):
        """Generate strategic recommendations based on analysis"""
        print("\n=== STRATEGIC RECOMMENDATIONS ===")
        
        # Get key insights
        overall_pl = self.pl_engine.get_pl_summary()
        performance_bands = self.scorer.get_performance_bands()
        segment_summary = self.segmenter.get_segment_summary()
        
        # Financial recommendations
        if overall_pl.loc['Total', 'net_margin'] < 0.05:
            print("Financial Strategy:")
            print("- Profit margins below target; focus on cost optimization and pricing strategy")
            print("- Review supplier contracts for food cost reduction")
            print("- Optimize staffing levels during low-demand periods")
        
        # Operational recommendations
        critical_locations = (performance_bands == 'Critical').sum()
        if critical_locations > 0:
            print("\nOperational Strategy:")
            print(f"- {critical_locations} locations require immediate intervention")
            print("- Conduct operational audits for underperforming branches")
            print("- Consider restructuring or closure of consistently loss-making locations")
        
        # Growth recommendations
        champion_segments = segment_summary[segment_summary['segment_name'] == 'High-Performing Champions']
        if len(champion_segments) > 0:
            print("\nGrowth Strategy:")
            print("- Replicate successful practices from high-performing locations")
            print("- Focus expansion efforts on CBD and high-traffic locations")
            print("- Invest in marketing during peak holiday seasons")
        
        # Segmentation recommendations
        print("\nSegmentation Strategy:")
        print("- Tailor management approaches to each performance segment")
        print("- High-performers: Focus on sustaining excellence and scaling best practices")
        print("- Efficient operators: Leverage for knowledge sharing and process optimization")
        print("- Underperformers: Develop targeted improvement plans or consider divestment")
    
    def generate_executive_summary(self):
        """Generate complete executive summary report"""
        print("EXECUTIVE SUMMARY - P&L AND KPI ANALYSIS")
        print("=" * 50)
        
        self.generate_financial_summary()
        self.generate_kpi_dashboard()
        self.generate_performance_insights()
        self.generate_segmentation_analysis()
        self.generate_strategic_recommendations()
        
        print("\n" + "=" * 50)
        print("END OF EXECUTIVE SUMMARY")

# Generate executive report
executive_report = ExecutiveReport(pl_engine, kpi_engine, scorer, segmenter)
executive_report.generate_executive_summary()